# 🔥 Лабораторная работа 4. PyTorch изнутри

Цель: разобрать основные объекты PyTorch как понятные строительные блоки.

Мы изучим Tensor, `nn.Module`, `nn.Linear`, `forward()`, параметры, `autograd`, режимы `train()` / `eval()` и Shapes внутри модели.


# 1. Импорт PyTorch

In [ ]:
import torch
import torch.nn as nn

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)

print("PyTorch version:", torch.__version__)

# 2. Первый Tensor

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])

print(x)
print("shape:", x.shape)
print("dtype:", x.dtype)
print("device:", x.device)

# 3. Tensor-матрица

In [ ]:
matrix = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
])

print(matrix)
print("shape:", matrix.shape)
print("ndim:", matrix.ndim)

# 4. Разные dtype

In [ ]:
float_tensor = torch.tensor([1.0, 2.0])
int_tensor = torch.tensor([1, 2])
bool_tensor = torch.tensor([True, False])

print(float_tensor.dtype)
print(int_tensor.dtype)
print(bool_tensor.dtype)

# 5. Device

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])

print("Текущий device:", x.device)
print("CUDA доступна:", torch.cuda.is_available())

# 6. Создаём nn.Linear(3, 4)

In [ ]:
layer = nn.Linear(3, 4)

print(layer)
print("weight shape:", layer.weight.shape)
print("bias shape:", layer.bias.shape)

# 7. Считаем параметры слоя

In [ ]:
print("Weights:", layer.weight.numel())
print("Bias:", layer.bias.numel())
print("Всего:", layer.weight.numel() + layer.bias.numel())

# 8. Передаём Tensor через Linear

In [ ]:
sample = torch.tensor([[1.0, 0.0, 1.0]])
output = layer(sample)

print("Input shape:", sample.shape)
print("Output shape:", output.shape)
print(output)

# 9. Создаём собственную модель

In [ ]:
class TrafficLightNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(3, 4)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(4, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x


torch.manual_seed(RANDOM_SEED)
model = TrafficLightNetwork()
print(model)

# 10. model(x) и forward()

In [ ]:
x = torch.tensor([[0.0, 0.0, 1.0]])

output_normal = model(x)
output_direct = model.forward(x)

print("model(x):", output_normal)
print("model.forward(x):", output_direct)

# 11. parameters()

In [ ]:
for i, parameter in enumerate(model.parameters()):
    print(i, parameter.shape, parameter.requires_grad)

# 12. named_parameters()

In [ ]:
for name, parameter in model.named_parameters():
    print(name, "|", parameter.shape, "| requires_grad =", parameter.requires_grad)

# 13. Считаем параметры модели

In [ ]:
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Всего параметров:", total)
print("Обучаемых параметров:", trainable)

# 14. Проходим по сети слой за слоем

In [ ]:
x = torch.tensor([[0.0, 0.0, 1.0]])

print("INPUT:", x.shape)

hidden = model.layer1(x)
print("AFTER LINEAR 1:", hidden.shape)
print(hidden)

activated = model.relu(hidden)
print("AFTER RELU:", activated.shape)
print(activated)

logit = model.layer2(activated)
print("AFTER LINEAR 2:", logit.shape)
print(logit)

# 15. Проверяем совпадение с model(x)

In [ ]:
normal_output = model(x)

print("Ручной проход:", logit)
print("model(x):", normal_output)
print("Совпадают:", torch.allclose(logit, normal_output))

# 16. nn.Sequential

In [ ]:
sequential_model = nn.Sequential(
    nn.Linear(3, 4),
    nn.ReLU(),
    nn.Linear(4, 1),
)

print(sequential_model)
print("Это nn.Module:", isinstance(sequential_model, nn.Module))

# 17. До backward() gradients отсутствуют

In [ ]:
for name, parameter in model.named_parameters():
    print(name, "grad =", parameter.grad)

# 18. Выполняем backward()

In [ ]:
target = torch.tensor([[1.0]])
loss_function = nn.BCEWithLogitsLoss()

logit = model(x)
loss = loss_function(logit, target)

model.zero_grad()
loss.backward()

print("Loss:", loss.item())

for name, parameter in model.named_parameters():
    print(name, "grad shape =", parameter.grad.shape)

# 19. Autograd на простом примере

In [ ]:
value = torch.tensor(2.0, requires_grad=True)
result = value ** 2

result.backward()

print("x:", value.item())
print("y=x²:", result.item())
print("dy/dx:", value.grad.item())

# 20. train() и eval()

In [ ]:
model.train()
print("После train():", model.training)

model.eval()
print("После eval():", model.training)

# 21. torch.no_grad()

In [ ]:
model.eval()

with torch.no_grad():
    prediction = model(x)

print(prediction)
print("requires_grad:", prediction.requires_grad)

# 22. state_dict()

In [ ]:
for name, tensor in model.state_dict().items():
    print(name, tensor.shape)

# 23. Batch и Shapes

In [ ]:
batch = torch.tensor([
    [0.0, 0.0, 1.0],
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 1.0],
    [1.0, 1.0, 0.0],
])

print("Input:", batch.shape)

hidden = model.layer1(batch)
print("After Linear 1:", hidden.shape)

activated = model.relu(hidden)
print("After ReLU:", activated.shape)

output = model.layer2(activated)
print("After Linear 2:", output.shape)

# 24. Практическая модель 4 → 8 → 2

In [ ]:
class PracticeNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 2),
        )

    def forward(self, x):
        return self.network(x)


practice_model = PracticeNetwork()

print(practice_model)

for name, parameter in practice_model.named_parameters():
    print(name, parameter.shape)

print(
    "Всего параметров:",
    sum(p.numel() for p in practice_model.parameters())
)

# 25. 📌 Что нужно запомнить

```text
Tensor
↓
Layer
↓
Tensor
↓
Activation
↓
Tensor
↓
Layer
↓
Tensor
```

Главные инструменты:

```python
tensor.shape
tensor.dtype
tensor.device

model.parameters()
model.named_parameters()

parameter.requires_grad
parameter.grad

model.train()
model.eval()

torch.no_grad()
```


# 26. ❓ Самопроверка

1. Что такое Tensor?
2. Что показывает `shape`?
3. Что делает `nn.Linear(3,4)`?
4. Что такое `nn.Module`?
5. Что делает `forward()`?
6. Что такое `nn.Sequential`?
7. Что возвращает `named_parameters()`?
8. Что означает `requires_grad`?
9. Что такое `autograd`?
10. Для чего нужен `torch.no_grad()`?
11. Почему важно следить за Shapes?


# 27. ➡️ Следующая глава

# Глава 5. MNIST — распознавание рукописных цифр

Следующим шагом перейдём к настоящему Dataset изображений.
